# PyPilot: LoRA Fine-tuning for Qwen2.5-Coder-7B

This notebook fine-tunes Qwen2.5-Coder-7B on the LeetCode dataset using LoRA (Low-Rank Adaptation) for parameter-efficient training.

**Requirements:** Google Colab with GPU (T4 or better, A100 recommended)

## 1. Setup & Install Dependencies

In [ ]:
# Install required packages
!pip install -q torch transformers datasets peft accelerate bitsandbytes trl huggingface_hub

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU available! Go to Runtime > Change runtime type > GPU")

## 2. Load Dataset from HuggingFace

In [ ]:
from datasets import load_dataset

# Load LeetCode dataset from HuggingFace
dataset = load_dataset("newfacade/LeetCodeDataset")

print(f"Train samples: {len(dataset['train'])}")
print(f"Test samples: {len(dataset['test'])}")

# Preview a sample
sample = dataset['train'][0]
print(f"\nSample task: {sample['task_id']}")
print(f"Query (first 200 chars): {sample['query'][:200]}...")
print(f"Completion (first 200 chars): {sample['completion'][:200]}...")

## 3. Format Dataset for Training

In [ ]:
# Qwen chat format tokens
CHAT_USER = "<|im_start|>user\n"
CHAT_ASSISTANT = "<|im_start|>assistant\n"
CHAT_END = "<|im_end|>"

def format_example(example: dict) -> dict:
    """
    Format a single example into Qwen chat format.
    
    The dataset has:
      - prompt: problem description + starter code
      - query: the actual user query
      - completion: the solution code
    """
    prompt = (example.get("prompt") or "").strip()
    query = (example.get("query") or "").strip()
    
    if prompt and query:
        instruction = prompt + "\n\n" + query
    elif query:
        instruction = query
    else:
        instruction = prompt
    
    response = example["completion"]
    
    # Qwen chat format
    text = f"{CHAT_USER}{instruction}{CHAT_END}\n{CHAT_ASSISTANT}{response}{CHAT_END}"
    
    return {"text": text}

# Format the dataset
print("Formatting dataset...")
train_dataset = dataset["train"].map(
    format_example,
    remove_columns=dataset["train"].column_names,
    desc="Formatting train",
)
eval_dataset = dataset["test"].map(
    format_example,
    remove_columns=dataset["test"].column_names,
    desc="Formatting eval",
)

print(f"Formatted {len(train_dataset)} train, {len(eval_dataset)} eval samples")
print(f"\nExample formatted text (first 500 chars):\n{train_dataset[0]['text'][:500]}...")

## 4. Load Model with 4-bit Quantization

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "Qwen/Qwen2.5-Coder-7B"

# 4-bit quantization config for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    padding_side="right",
)

# Set pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Loading model with 4-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

print("Model loaded successfully!")

## 5. Configure LoRA

In [ ]:
# LoRA configuration
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                    # LoRA rank (higher = more capacity, more memory)
    lora_alpha=16,          # LoRA alpha (scaling factor)
    lora_dropout=0.05,      # Dropout for regularization
    target_modules=[        # Qwen2.5 attention and MLP modules
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    bias="none",
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

## 6. Training Configuration

In [ ]:
from trl import SFTConfig, SFTTrainer

# Training hyperparameters
OUTPUT_DIR = "./qwen-lora-leetcode"
EPOCHS = 3
BATCH_SIZE = 2
GRADIENT_ACCUMULATION = 8
LEARNING_RATE = 2e-4  # Important: Use 2e-4, NOT 1e-3 (too high causes mode collapse)
MAX_SEQ_LENGTH = 2048

# Check bf16 support
supports_bf16 = torch.cuda.is_bf16_supported()
print(f"bfloat16 supported: {supports_bf16}")

# SFT training config
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    bf16=supports_bf16,
    fp16=not supports_bf16,
    max_grad_norm=0.3,
    report_to="none",
    push_to_hub=False,
    
    # SFT specific
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
)

print(f"Training config ready. Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION}")

## 7. Train the Model

In [ ]:
# Initialize trainer
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

print("Starting training...")
print(f"Train samples: {len(train_dataset)}")
print(f"Eval samples: {len(eval_dataset)}")

In [ ]:
# Train!
trainer.train()

## 8. Save the Model

In [ ]:
# Save final model
FINAL_MODEL_PATH = f"{OUTPUT_DIR}/final"
print(f"Saving model to {FINAL_MODEL_PATH}...")

trainer.save_model(FINAL_MODEL_PATH)
tokenizer.save_pretrained(FINAL_MODEL_PATH)

print("Model saved!")

## 9. Test the Fine-tuned Model

In [ ]:
# Load and merge the LoRA model for inference
from peft import PeftModel

print("Merging LoRA weights for inference...")
model = model.merge_and_unload()
print("Model ready for inference!")

In [ ]:
def generate_solution(problem: str, max_new_tokens: int = 512) -> str:
    """Generate a solution for a coding problem."""
    prompt = f"{CHAT_USER}{problem}{CHAT_END}\n{CHAT_ASSISTANT}"
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Build stop token IDs
    stop_ids = [tokenizer.eos_token_id]
    for token in ["<|im_end|>", "<|file_sep|>", "<|endoftext|>"]:
        tid = tokenizer.convert_tokens_to_ids(token)
        if tid and tid != tokenizer.unk_token_id:
            stop_ids.append(tid)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=stop_ids,
        )
    
    generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return generated

In [ ]:
# Test on a sample problem
test_problem = """You are an expert Python programmer. Write a function that takes a list of integers and returns the two numbers that add up to a target sum.

Example:
Input: nums = [2, 7, 11, 15], target = 9
Output: [0, 1] (because nums[0] + nums[1] = 2 + 7 = 9)

Write a Python class Solution with method twoSum(self, nums: List[int], target: int) -> List[int]"""

print("Generating solution...")
solution = generate_solution(test_problem)
print("\n" + "="*60)
print("GENERATED SOLUTION:")
print("="*60)
print(solution)

## 10. Download Model (Optional)

In [ ]:
# Zip and download the model
!zip -r qwen-lora-leetcode-final.zip {FINAL_MODEL_PATH}

from google.colab import files
files.download('qwen-lora-leetcode-final.zip')

## 11. Push to HuggingFace Hub (Optional)

In [ ]:
# Uncomment and run to push to HuggingFace Hub
# from huggingface_hub import login
# login()  # Enter your HuggingFace token

# HUB_MODEL_ID = "your-username/qwen-lora-leetcode"
# model.push_to_hub(HUB_MODEL_ID)
# tokenizer.push_to_hub(HUB_MODEL_ID)
# print(f"Model pushed to https://huggingface.co/{HUB_MODEL_ID}")